# Peramalan Produksi Padi per Pulau dengan SARIMAX
Alur: **Data → EDA → Uji Stasioneritas & Penanganan → Identifikasi Model → Evaluasi (holdout 12 bulan) → Diagnostik → Peramalan 12 bulan**

Cakupan: satu pulau yang dipilih lewat `PULAU_PILIH` di sel Pengaturan (bawaan: **Sulawesi**). Provinsi pulau tersebut disaring otomatis dari file lengkap 38 provinsi, ditambah satu deret total pulau.

Cara pakai: jalankan *Runtime → Run all*, lalu unggah `Data_Padi.xlsx` saat diminta.

## 0. Pengaturan

In [ ]:
import warnings, itertools
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)

# ---- PARAMETER (ubah sesuai kebutuhan) ----
PULAU_PILIH = "Sulawesi"   # pilihan: Sumatera, Jawa, Bali-Nusa Tenggara, Kalimantan, Sulawesi, Maluku, Papua
PETA_PULAU = {
    "Sumatera": ["ACEH", "SUMATERA UTARA", "SUMATERA BARAT", "RIAU", "JAMBI", "SUMATERA SELATAN", "BENGKULU", "LAMPUNG",
                 "KEPULAUAN BANGKA BELITUNG", "KEPULAUAN RIAU"],
    "Jawa": ["DKI JAKARTA", "JAWA BARAT", "JAWA TENGAH", "DI YOGYAKARTA", "JAWA TIMUR", "BANTEN"],
    "Bali-Nusa Tenggara": ["BALI", "NUSA TENGGARA BARAT", "NUSA TENGGARA TIMUR"],
    "Kalimantan": ["KALIMANTAN BARAT", "KALIMANTAN TENGAH", "KALIMANTAN SELATAN", "KALIMANTAN TIMUR", "KALIMANTAN UTARA"],
    "Sulawesi": ["SULAWESI UTARA", "GORONTALO", "SULAWESI TENGAH", "SULAWESI BARAT", "SULAWESI SELATAN", "SULAWESI TENGGARA"],
    "Maluku": ["MALUKU", "MALUKU UTARA"],
    "Papua": ["PAPUA BARAT", "PAPUA BARAT DAYA", "PAPUA", "PAPUA SELATAN", "PAPUA TENGAH", "PAPUA PENGUNUNGAN"],
}
PULAU = PETA_PULAU[PULAU_PILIH]
TOTAL = f"TOTAL {PULAU_PILIH.upper()}"
TARGET = "Y"                           # variabel yang diramal
EXOG = ["X", "Curah_Hujan_mm"]         # variabel eksogen (isi [] untuk SARIMA biasa)
S = 12                                 # periode musiman (bulanan)
H = 12                                 # horizon peramalan (bulan)
TEST = 12                              # panjang data uji (holdout)
USE_LOG = True                         # transformasi log untuk menstabilkan varians

## 1. Data

In [ ]:
from google.colab import files
up = files.upload()                    # pilih Data_Padi.xlsx
fname = list(up.keys())[0]
raw = pd.read_excel(fname)
raw.head()

In [ ]:
bulan = {"januari":1,"februari":2,"maret":3,"april":4,"mei":5,"juni":6,"juli":7,
         "agustus":8,"september":9,"oktober":10,"nopember":11,"november":11,"desember":12}
df = raw.copy()
df["Provinsi"] = df["Provinsi"].str.strip().str.upper()
df["bln"] = df["Bulan"].astype(str).str.strip().str.lower().map(bulan)
assert df["bln"].notna().all(), "Ada nama bulan yang tidak terbaca"
df["Tanggal"] = pd.to_datetime(dict(year=df["Tahun"], month=df["bln"], day=1))

hilang = sorted(set(PULAU) - set(df["Provinsi"]))
assert not hilang, f"Provinsi tidak ditemukan di file: {hilang}"
pl = df[df["Provinsi"].isin(PULAU)].copy()

# pivot per variabel: baris = waktu, kolom = provinsi
def pv(col):
    p = pl.pivot_table(index="Tanggal", columns="Provinsi", values=col, aggfunc="sum")
    p = p.asfreq("MS")
    p[TOTAL] = p[PULAU].sum(axis=1)
    return p

data = {c: pv(c) for c in [TARGET] + EXOG}
# curah hujan total = rata-rata (bukan penjumlahan)
if "Curah_Hujan_mm" in EXOG:
    ch = pl.pivot_table(index="Tanggal", columns="Provinsi", values="Curah_Hujan_mm").asfreq("MS")
    ch[TOTAL] = ch[PULAU].mean(axis=1)
    data["Curah_Hujan_mm"] = ch

# provinsi yang memuat nilai 0 pada Y tidak dimodelkan sendiri (log tidak terdefinisi); tetap ikut dalam total
ada_nol = [p for p in PULAU if (data[TARGET][p] == 0).any()]
MODEL_PROV = [p for p in PULAU if p not in ada_nol] + [TOTAL]
print("Pulau   :", PULAU_PILIH, "|", len(PULAU), "provinsi")
if ada_nol: print("Dikeluarkan dari pemodelan (ada nilai 0):", ada_nol)
print("Periode :", data[TARGET].index.min().date(), "s.d.", data[TARGET].index.max().date(),
      f"({len(data[TARGET])} bulan)")
print("Missing :", int(sum(d.isna().sum().sum() for d in data.values())))
data[TARGET].describe().T.round(0)

## 2. EDA

### 2.1 Ringkasan dan kualitas data

In [ ]:
ringkas = pd.DataFrame({
    "rata-rata": data[TARGET].mean(),
    "median": data[TARGET].median(),
    "std": data[TARGET].std(),
    "min": data[TARGET].min(),
    "maks": data[TARGET].max(),
    "CV (%)": data[TARGET].std() / data[TARGET].mean() * 100,
    "jumlah nol": (data[TARGET] == 0).sum(),
}).round(1)
ringkas

### 2.2 Plot deret waktu

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
for ax, p in zip(axes.ravel(), MODEL_PROV):
    data[TARGET][p].plot(ax=ax, lw=1.4)
    ax.set_title(p); ax.set_xlabel("")
fig.suptitle(f"{TARGET} bulanan per provinsi, Pulau {PULAU_PILIH}", y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

### 2.3 Pola musiman

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
for ax, p in zip(axes.ravel(), MODEL_PROV):
    s = data[TARGET][p]
    tab = s.groupby([s.index.month]).mean()
    ax.bar(tab.index, tab.values, color="#3b7dd8")
    ax.set_title(f"Rata-rata per bulan: {p}"); ax.set_xticks(range(1, 13))
plt.tight_layout(); plt.show()

# heatmap tahun x bulan untuk total pulau
t = data[TARGET][TOTAL]
hm = t.to_frame("v").assign(Tahun=t.index.year, Bulan=t.index.month).pivot(index="Tahun", columns="Bulan", values="v")
plt.figure(figsize=(11, 4)); sns.heatmap(hm/1e6, cmap="YlGn", annot=True, fmt=".1f", cbar_kws={"label": "juta"})
plt.title(f"{TOTAL}: pola tahun x bulan"); plt.show()

### 2.4 Dekomposisi STL dan kekuatan musiman/tren

In [ ]:
rows = []
for p in MODEL_PROV:
    s = data[TARGET][p]
    s = np.log(s) if USE_LOG else s
    r = STL(s, period=S, robust=True).fit()
    Ft = max(0, 1 - r.resid.var() / (r.trend + r.resid).var())
    Fs = max(0, 1 - r.resid.var() / (r.seasonal + r.resid).var())
    rows.append([p, round(Ft, 3), round(Fs, 3)])
    if p == TOTAL:
        r.plot(); plt.suptitle(f"Dekomposisi STL: {TOTAL}", y=1.02); plt.show()
pd.DataFrame(rows, columns=["Provinsi", "Kekuatan tren", "Kekuatan musiman"]).set_index("Provinsi")

Nilai kekuatan mendekati 1 berarti komponen tersebut dominan. Musiman kuat mendukung penggunaan komponen musiman (P, D, Q) pada SARIMAX.

### 2.5 Hubungan dengan variabel eksogen

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, p in zip(axes.ravel(), MODEL_PROV):
    ax.scatter(data["X"][p], data[TARGET][p], s=14, alpha=.7); ax.set_title(f"{p}: Y vs X"); ax.set_xlabel("X"); ax.set_ylabel("Y")
plt.tight_layout(); plt.show()

kor = {}
for p in MODEL_PROV:
    d = pd.concat([data[c][p].rename(c) for c in [TARGET] + EXOG], axis=1)
    kor[p] = d.corr()[TARGET].drop(TARGET)
    # korelasi silang curah hujan dengan Y pada lag 0..6
kor = pd.DataFrame(kor).T.round(3)
display(kor)

lagrows = {}
for p in MODEL_PROV:
    lagrows[p] = [data[TARGET][p].corr(data["Curah_Hujan_mm"][p].shift(l)) for l in range(0, 7)]
display(pd.DataFrame(lagrows, index=[f"lag {l}" for l in range(7)]).T.round(2))

**Perhatian**: bila `X` (misalnya luas panen) berkorelasi hampir 1 dengan `Y`, maka `Y ≈ produktivitas × X`. Model akan tampak sangat akurat
pada data historis, tetapi untuk meramal ke depan nilai `X` juga harus diketahui atau diramal. Akurasi peramalan akhirnya bergantung pada
akurasi peramalan `X`. Bagian 6 memperlihatkan perbandingan dengan dan tanpa eksogen agar kamu bisa menilai sendiri.

## 3. Uji Stasioneritas dan Penanganannya

Dua uji dipakai bersama karena hipotesisnya berlawanan:
- **ADF**: H0 = ada akar unit (tidak stasioner). Stasioner bila p < 0,05.
- **KPSS**: H0 = stasioner. Stasioner bila p > 0,05.

In [ ]:
def uji(s, reg="c"):
    s = s.dropna()
    a = adfuller(s, autolag="AIC", regression=reg)
    k = kpss(s, regression=reg, nlags="auto")
    stas = (a[1] < .05) and (k[1] > .05)
    return {"ADF stat": round(a[0], 3), "ADF p": round(a[1], 4), "KPSS stat": round(k[0], 3), "KPSS p": round(k[1], 4),
            "Stasioner?": "Ya" if stas else "Tidak"}

def transform(s):
    return np.log(s) if USE_LOG else s.copy()

tahap = {}
hasil = []
for p in MODEL_PROV:
    z = transform(data[TARGET][p])
    tahap[p] = {"asli": z}
    r0 = uji(z); hasil.append({"Provinsi": p, "Tahap": "Level (log)" if USE_LOG else "Level", **r0})
    z1 = z.diff(1).dropna();           tahap[p]["d1"] = z1
    r1 = uji(z1); hasil.append({"Provinsi": p, "Tahap": "Diff biasa (d=1)", **r1})
    zs = z.diff(S).dropna();           tahap[p]["D1"] = zs
    r2 = uji(zs); hasil.append({"Provinsi": p, "Tahap": f"Diff musiman (D=1, s={S})", **r2})
    zb = z.diff(S).diff(1).dropna();   tahap[p]["d1D1"] = zb
    r3 = uji(zb); hasil.append({"Provinsi": p, "Tahap": "Diff musiman + biasa (d=1, D=1)", **r3})
tabel_stas = pd.DataFrame(hasil).set_index(["Provinsi", "Tahap"])
tabel_stas

### 3.1 Menentukan d dan D secara otomatis

In [ ]:
def tentukan_dD(z, S=12, maxd=2):
    # 1) kekuatan musiman menentukan D: bila kuat, D=1 (maksimal 1)
    r = STL(z, period=S, robust=True).fit()
    Fs = max(0, 1 - r.resid.var() / (r.seasonal + r.resid).var())
    D = 1 if Fs > 0.64 else 0           # ambang Wang, Smith & Hyndman (2006)
    x = z.diff(S).dropna() if D else z
    # 2) d dari ADF + KPSS pada deret setelah differencing musiman
    d = 0
    while d <= maxd:
        xx = x.diff(d).dropna() if d else x
        a = adfuller(xx, autolag="AIC")[1]; k = kpss(xx, regression="c", nlags="auto")[1]
        if a < .05 and k > .05: break
        d += 1
    return d, D, round(Fs, 3)

orde_dD = {}
for p in MODEL_PROV:
    d, D, Fs = tentukan_dD(tahap[p]["asli"], S)
    orde_dD[p] = (d, D)
pd.DataFrame(orde_dD, index=["d", "D"]).T.assign(catatan="dipakai pada model")

Hasil ini hanya titik awal. Hasil ADF/KPSS bisa saling bertentangan pada sampel pendek (n = 104), sehingga pemilihan akhir tetap dikonfirmasi lewat AIC dan diagnostik residual.

### 3.2 Visual sebelum dan sesudah differencing

In [ ]:
p = TOTAL
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for ax, (k, ttl) in zip(axes.ravel(), [("asli", "Level (log)"), ("d1", "Diff biasa"), ("D1", f"Diff musiman (s={S})"), ("d1D1", "Diff musiman + biasa")]):
    tahap[p][k].plot(ax=ax); ax.axhline(tahap[p][k].mean(), color="r", ls="--", lw=.8); ax.set_title(ttl)
plt.tight_layout(); plt.show()

z = tahap[p]["d1D1"] if orde_dD[p] == (1, 1) else tahap[p]["D1"]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(z, lags=36, ax=ax[0]); plot_pacf(z, lags=36, ax=ax[1], method="ywm")
ax[0].set_title(f"ACF, {p} (setelah differencing terpilih)"); ax[1].set_title("PACF")
plt.show()

Gunakan ACF/PACF di atas sebagai petunjuk: lonjakan pada lag 1-2 menunjukkan p, q non-musiman; lonjakan pada lag 12, 24 menunjukkan P, Q musiman.

## 4. Identifikasi dan Estimasi Model

Pencarian grid kecil atas (p, q, P, Q) dengan d dan D dari bagian 3, dipilih berdasarkan **AIC** pada data latih.
Dilakukan dua varian: **SARIMAX dengan eksogen** dan **SARIMA tanpa eksogen** sebagai pembanding.

In [ ]:
def split(p, use_exog):
    y = transform(data[TARGET][p])
    ex = pd.concat([data[c][p].rename(c) for c in EXOG], axis=1) if (use_exog and EXOG) else None
    if ex is not None and USE_LOG:
        ex = ex.copy()
        if "X" in ex: ex["X"] = np.log(ex["X"])
    return y, ex

def fit(y, ex, order, sorder):
    m = SARIMAX(y, exog=ex, order=order, seasonal_order=sorder, trend="n" if (order[1] or sorder[1]) else "c",
                enforce_stationarity=False, enforce_invertibility=False)
    return m.fit(disp=False, maxiter=200)

def grid_search(y, ex, d, D, pmax=2, qmax=2, Pmax=1, Qmax=1):
    best, tabel = None, []
    for p_, q_, P_, Q_ in itertools.product(range(pmax+1), range(qmax+1), range(Pmax+1), range(Qmax+1)):
        try:
            r = fit(y, ex, (p_, d, q_), (P_, D, Q_, S))
            tabel.append(((p_, d, q_), (P_, D, Q_, S), r.aic, r.bic))
            if best is None or r.aic < best[2]: best = ((p_, d, q_), (P_, D, Q_, S), r.aic)
        except Exception:
            continue
    return best, pd.DataFrame(tabel, columns=["order", "seasonal", "AIC", "BIC"]).sort_values("AIC")

pilihan = {}
for p in MODEL_PROV:
    d, D = orde_dD[p]
    for var, ue in [("SARIMAX", True), ("SARIMA", False)]:
        y, ex = split(p, ue)
        ytr = y.iloc[:-TEST]; extr = ex.iloc[:-TEST] if ex is not None else None
        best, tab = grid_search(ytr, extr, d, D)
        pilihan[(p, var)] = best
        print(f"{p:15s} {var:8s} -> order={best[0]} seasonal={best[1]} AIC={best[2]:.1f}")

## 5. Evaluasi pada Data Uji (12 bulan terakhir)

In [ ]:
def metrik(a, f):
    a, f = np.asarray(a), np.asarray(f)
    return {"MAPE (%)": np.mean(np.abs((a - f) / a)) * 100,
            "RMSE": np.sqrt(np.mean((a - f) ** 2)),
            "MAE": np.mean(np.abs(a - f))}

def kembali(v):
    return np.exp(v) if USE_LOG else v

eval_rows, fc_test, model_train = [], {}, {}
for p in MODEL_PROV:
    yfull = data[TARGET][p]
    atest = yfull.iloc[-TEST:]
    for var, ue in [("SARIMAX", True), ("SARIMA", False)]:
        y, ex = split(p, ue)
        order, sorder, _ = pilihan[(p, var)]
        r = fit(y.iloc[:-TEST], ex.iloc[:-TEST] if ex is not None else None, order, sorder)
        pred = r.get_forecast(TEST, exog=ex.iloc[-TEST:] if ex is not None else None).predicted_mean
        pred = kembali(pred)
        fc_test[(p, var)] = pred; model_train[(p, var)] = r
        eval_rows.append({"Provinsi": p, "Model": var, **metrik(atest, pred)})
    # pembanding naif musiman
    naif = yfull.shift(S).iloc[-TEST:]
    eval_rows.append({"Provinsi": p, "Model": "Naif musiman", **metrik(atest, naif)})

evaluasi = pd.DataFrame(eval_rows).set_index(["Provinsi", "Model"]).round(2)
evaluasi

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
for ax, p in zip(axes.ravel(), MODEL_PROV):
    yfull = data[TARGET][p]
    yfull.iloc[-36:].plot(ax=ax, color="k", label="Aktual")
    fc_test[(p, "SARIMAX")].plot(ax=ax, color="tab:red", label="SARIMAX")
    fc_test[(p, "SARIMA")].plot(ax=ax, color="tab:blue", ls="--", label="SARIMA")
    ax.set_title(p); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

Bacalah tabel di atas dengan kritis: bila SARIMAX **tidak** lebih baik dari SARIMA (atau dari naif musiman) pada data uji, tambahan eksogen belum membantu
dan model yang lebih sederhana lebih layak dipakai.

## 6. Diagnostik Residual

In [ ]:
def diagnostik(r, nama):
    res = r.resid.iloc[max(S, 13):]
    lb = acorr_ljungbox(res, lags=[12, 24], return_df=True)
    jb = jarque_bera(res)
    return {"Provinsi": nama, "LB p (lag 12)": round(lb["lb_pvalue"].iloc[0], 3), "LB p (lag 24)": round(lb["lb_pvalue"].iloc[1], 3),
            "JB p (normalitas)": round(jb[1], 3)}

dg = pd.DataFrame([diagnostik(model_train[(p, "SARIMAX")], p) for p in MODEL_PROV]).set_index("Provinsi")
display(dg)

p = TOTAL
model_train[(p, "SARIMAX")].plot_diagnostics(figsize=(13, 8)); plt.suptitle(f"Diagnostik residual SARIMAX: {p}", y=1.02); plt.show()

Ljung-Box p > 0,05 berarti residual tidak berautokorelasi (model sudah menangkap pola). Jarque-Bera p > 0,05 berarti residual mendekati normal. Pelanggaran ringan normalitas tidak menggugurkan model, tetapi selang kepercayaan perlu dibaca hati-hati.

## 7. Peramalan 12 Bulan ke Depan

Model dilatih ulang pada **seluruh data**, lalu meramal `H` bulan setelah data terakhir. Variabel eksogen di masa depan tidak diketahui,
sehingga diisi dengan **rata-rata bulanan 3 tahun terakhir** (pola musiman). Ini asumsi sederhana dan transparan; bila kamu punya
proyeksi resmi untuk luas panen atau curah hujan (misalnya prakiraan BMKG), ganti pada sel di bawah.

In [ ]:
def exog_masa_depan(p, idx_future):
    out = {}
    for c in EXOG:
        s = data[c][p]
        rata = s.iloc[-36:].groupby(s.iloc[-36:].index.month).mean()
        out[c] = [rata[m] for m in idx_future.month]
    ef = pd.DataFrame(out, index=idx_future)
    if USE_LOG and "X" in ef: ef["X"] = np.log(ef["X"])
    return ef

last = data[TARGET].index[-1]
fidx = pd.date_range(last + pd.offsets.MonthBegin(1), periods=H, freq="MS")

hasil_fc, model_final = {}, {}
for p in MODEL_PROV:
    var = "SARIMAX" if EXOG else "SARIMA"
    y, ex = split(p, bool(EXOG))
    order, sorder, _ = pilihan[(p, var)]
    r = fit(y, ex, order, sorder)
    ef = exog_masa_depan(p, fidx) if EXOG else None
    f = r.get_forecast(H, exog=ef)
    ci = f.conf_int(alpha=.05)
    hasil_fc[p] = pd.DataFrame({"Ramalan": kembali(f.predicted_mean), "Bawah 95%": kembali(ci.iloc[:, 0]), "Atas 95%": kembali(ci.iloc[:, 1])})
    model_final[p] = r

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
for ax, p in zip(axes.ravel(), MODEL_PROV):
    data[TARGET][p].iloc[-48:].plot(ax=ax, color="k", label="Aktual")
    hasil_fc[p]["Ramalan"].plot(ax=ax, color="tab:red", label="Ramalan")
    ax.fill_between(fidx, hasil_fc[p]["Bawah 95%"], hasil_fc[p]["Atas 95%"], color="tab:red", alpha=.2, label="95% CI")
    ax.set_title(p); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

hasil_fc[TOTAL].round(0)

In [ ]:
# ringkasan: rata-rata ramalan vs rata-rata 12 bulan terakhir
rg = []
for p in MODEL_PROV:
    a = data[TARGET][p].iloc[-12:].sum(); b = hasil_fc[p]["Ramalan"].sum()
    rg.append({"Provinsi": p, "Total 12 bln terakhir (aktual)": a, "Total 12 bln ke depan (ramalan)": b, "Perubahan (%)": (b / a - 1) * 100})
pd.DataFrame(rg).set_index("Provinsi").round(1)

## 8. Simpan Hasil

In [ ]:
with pd.ExcelWriter(f"hasil_sarimax_{PULAU_PILIH.lower()}.xlsx") as w:
    tabel_stas.to_excel(w, sheet_name="Uji_Stasioner")
    pd.DataFrame(orde_dD, index=["d", "D"]).T.to_excel(w, sheet_name="Orde_dD")
    evaluasi.to_excel(w, sheet_name="Evaluasi_Holdout")
    dg.to_excel(w, sheet_name="Diagnostik")
    for p, d_ in hasil_fc.items():
        d_.to_excel(w, sheet_name=f"Ramalan_{p[:20]}")
files.download(f"hasil_sarimax_{PULAU_PILIH.lower()}.xlsx")